# Module 4: Search Evaluation



In [1]:
import pandas as pd

In [2]:
filepath = "./data/ground_truth.csv"

In [3]:
df_ground_truth = pd.read_csv(filepath)

In [4]:
df_ground_truth.head()

,question,document
0,Can I still join the course if I'm late to it?,74eb249bbf
1,"If I join the course now, can I still get a ce...",74eb249bbf
2,Do I need to finish the project before submiss...,74eb249bbf
3,"Is it too late to start this course, or can I ...",74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf


In [5]:
ground_truth = df_ground_truth.to_dict(orient="records")

Now that we have our search data ready, we need to index our documents so that we will be able to search them.

In [7]:
from ingest import load_faq_data, build_index

In [8]:
documents = load_faq_data()
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [ ]:
def text_search(query:str):
    boost_dict = {"question": 3.0, "section": 0.5}
    # If we were using all docs we could also define a filter dict here to add to the search that is returned.
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [11]:
from minsearch import Index

In [21]:
ground_truth[0]

{'question': "Can I still join the course if I'm late to it?",
 'document': '74eb249bbf'}

In [15]:
q = ground_truth[0]["question"]

In [16]:
q

"Can I still join the course if I'm late to it?"

In [18]:
results = text_search(q)

In [20]:
results[0]["id"]

'74eb249bbf'

In [22]:
for res in results:
    if res["id"] == ground_truth[0]["document"]:
        print(res)

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [23]:
q = ground_truth[0]
q

{'question': "Can I still join the course if I'm late to it?",
 'document': '74eb249bbf'}

In [24]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [25]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False


In [26]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

In [27]:
relevance

[1, 0, 0, 0, 0]

## Search across all search terms

In [ ]:
def compute_relevant_text(q: dict[str]):
    # get id of document used to generate the query
    doc_id = q["document"]
    # search q* term against original docs
    results = text_search(query=q["question"])

    relevance = []

    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

In [30]:
from tqdm.auto import tqdm

In [31]:
def compute_relevance_total_text(ground_truth: list[dict]):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevant_text(q)
        relevance_total.append(relevance)
    
    return relevance_total

In [32]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/515 [00:00<?, ?it/s]

In [33]:
relevance

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,